# Notebook 3: Causal Price Elasticity Estimation

**Self-contained.** This notebook loads the dataset directly from HuggingFace.

We estimate the **average causal effect** of price on quantity using the **Partially Linear Regression (PLR)** model from Double Machine Learning (Chernozhukov et al., 2018).

**Model:**
$$Q_{it} = \theta \cdot P_{it} + g(X_{it}) + \varepsilon_{it}$$
$$P_{it} = m(X_{it}) + v_{it}$$

$\theta$ is the causal price elasticity; $g$ and $m$ are estimated with machine learning.

**No cross-fitting** – nuisance models ($\hat{g}$, $\hat{m}$) are trained on the **train split**; the causal parameter $\theta$ is estimated from **test-split residuals** only. This avoids overfitting bias without $k$-fold cross-fitting.

**No heterogeneity analysis** – we estimate a single average treatment effect.

In [ ]:
import re
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

from lightgbm import LGBMRegressor
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import pairwise_distances, r2_score

palette = sns.color_palette("colorblind")
confidence_level = 0.90

## 1. Load and Prepare Data

In [ ]:
ds = datasets.load_dataset("janteichertkluge/demand-analysis-repro")
df_train = ds["train"].to_pandas()
df_test  = ds["test"].to_pandas()

df_train = df_train.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"})
df_test  = df_test.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"})
df_train["date"] = df_train["date"].astype(str)
df_test["date"]  = df_test["date"].astype(str)

print(f"Train: {df_train.shape}  |  Test: {df_test.shape}")

### 1a. Embeddings → PCA and Cluster Similarities

In [ ]:
emb_cols = sorted([c for c in df_train.columns if c.startswith("emb_")])
print(f"Embedding dimensions: {len(emb_cols)}")

emb_tr = df_train[emb_cols].values.astype(float)
emb_te = df_test[emb_cols].values.astype(float)

global_mean = emb_tr.mean(axis=0)
emb_tr_n = normalize(emb_tr - global_mean, axis=1)
emb_te_n = normalize(emb_te - global_mean, axis=1)

# PCA (fit on train)
pca = PCA(n_components=5, random_state=42)
pca.fit(emb_tr_n)
pca_cols = [f"pca_{i}" for i in range(5)]
df_train[pca_cols] = pca.transform(emb_tr_n)
df_test[pca_cols]  = pca.transform(emb_te_n)

# K-Means cluster similarities (fit on train)
km = KMeans(n_clusters=5, random_state=42, n_init=50)
km.fit(emb_tr_n)
sim_cols = [f"sim_cluster_{i}" for i in range(5)]
for j, c in enumerate(km.cluster_centers_):
    df_train[sim_cols[j]] = 1.0 - pairwise_distances(emb_tr_n, c.reshape(1,-1), metric="cosine").ravel()
    df_test[sim_cols[j]]  = 1.0 - pairwise_distances(emb_te_n, c.reshape(1,-1), metric="cosine").ravel()

print("Embedding features ready.")

### 1b. Lagged Variables and Dummies

In [ ]:
def add_lags(df, vars_, n=1):
    df = df.sort_values(["ASIN", "date"]).copy()
    for v in vars_:
        for i in range(1, n+1):
            df[f"{v}_lag{i}"] = df.groupby("ASIN")[v].shift(i)
    return df

df_train = add_lags(df_train, ["Q_t", "P_bb_t", "REVIEW_COUNT", "RATING"])
df_test  = add_lags(df_test,  ["Q_t", "P_bb_t", "REVIEW_COUNT", "RATING"])

if "subcat_aggregated" in df_train.columns:
    sub_tr = pd.get_dummies(df_train["subcat_aggregated"], prefix="sub", drop_first=True).astype(int)
    sub_te = pd.get_dummies(df_test["subcat_aggregated"],  prefix="sub", drop_first=True).astype(int)
    sub_te = sub_te.reindex(columns=sub_tr.columns, fill_value=0)
    df_train[sub_tr.columns] = sub_tr.values
    df_test[sub_te.columns]  = sub_te.values
    subcat_cols = sub_tr.columns.tolist()
else:
    subcat_cols = []

time_tr = pd.get_dummies(df_train["date"], prefix="t", drop_first=True).astype(int)
time_te = pd.get_dummies(df_test["date"],  prefix="t", drop_first=True).astype(int)
time_te = time_te.reindex(columns=time_tr.columns, fill_value=0)
df_train[time_tr.columns] = time_tr.values
df_test[time_te.columns]  = time_te.values
time_cols = time_tr.columns.tolist()

df_train = df_train.dropna(subset=["Q_t_lag1", "P_bb_t_lag1"]).reset_index(drop=True)
df_test  = df_test.dropna(subset=["Q_t_lag1", "P_bb_t_lag1"]).reset_index(drop=True)

print(f"After lags  →  Train: {df_train.shape}  |  Test: {df_test.shape}")

## 2. Define Variables and Control Specifications

In [ ]:
outcome   = "Q_t"
treatment = "P_bb_t"

lag_controls = ["Q_t_lag1", "P_bb_t_lag1"]
cont_controls_candidates = [
    "RATING_lag1", "REVIEW_COUNT_lag1",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
    "Lightning Deals: Upcoming Deal",
    "Buy Box: Is FBA",
]
cont_controls = [c for c in cont_controls_candidates if c in df_train.columns]
base_controls = lag_controls + cont_controls + subcat_cols + time_cols

# Control specifications to compare
specs = {
    "Lagged vars only":      lag_controls,
    "Tabular controls":      base_controls,
    "Tabular + PCA":         base_controls + pca_cols,
    "Tabular + Sim":         base_controls + sim_cols,
    "Tabular + Sim + PCA":   base_controls + sim_cols + pca_cols,
}
print(f"Base controls: {len(base_controls)}  |  PCA: {len(pca_cols)}  |  Sim: {len(sim_cols)}")

## 3. DML Estimator (No Cross-Fitting)

**Algorithm:**
1. Fit nuisance models on **train**: $\hat{g}(X)$ (for outcome) and $\hat{m}(X)$ (for treatment).
2. Compute residuals on **test**: $\tilde{Q}_i = Q_i - \hat{g}(X_i)$, $\tilde{P}_i = P_i - \hat{m}(X_i)$.
3. Estimate $\hat{\theta}$ by OLS of $\tilde{Q}$ on $\tilde{P}$ (clustered SE by ASIN).

In [ ]:
def clean_cols(df):
    df = df.copy()
    df.columns = [re.sub(r"[^A-Za-z0-9_]+", "", c) for c in df.columns]
    return df

def get_X(df, controls):
    cols = [c for c in controls if c in df.columns]
    return clean_cols(df[cols].fillna(0))


def dml_no_crossfit(df_train, df_test, controls, outcome, treatment,
                    confidence_level=0.90, label=""):
    """
    DML/PLR estimator without cross-fitting.
    Nuisance: LightGBM, fitted on df_train.
    Second stage: OLS on df_test residuals, clustered by ASIN.
    """
    X_tr = get_X(df_train, controls)
    X_te = get_X(df_test,  controls)
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

    # First stage: nuisance models
    ml_g = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    ml_g.fit(X_tr, df_train[outcome])

    ml_m = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    ml_m.fit(X_tr, df_train[treatment])

    # Residuals on test set
    Q_tilde = df_test[outcome].values   - ml_g.predict(X_te)
    P_tilde = df_test[treatment].values - ml_m.predict(X_te)

    r2_g = r2_score(df_test[outcome],   ml_g.predict(X_te))
    r2_m = r2_score(df_test[treatment], ml_m.predict(X_te))

    # Second stage: OLS with clustered SE
    df_r = pd.DataFrame({"Q_tilde": Q_tilde, "P_tilde": P_tilde,
                          "ASIN": df_test["ASIN"].values})
    ols = sm.OLS(df_r["Q_tilde"], df_r[["P_tilde"]]).fit(
        cov_type="cluster", cov_kwds={"groups": df_r["ASIN"]}
    )

    alpha  = 1 - confidence_level
    theta  = float(ols.params["P_tilde"])
    se     = float(ols.bse["P_tilde"])
    ci     = ols.conf_int(alpha=alpha)
    ci_lo  = float(ci.loc["P_tilde", 0])
    ci_hi  = float(ci.loc["P_tilde", 1])
    pval   = float(ols.pvalues["P_tilde"])

    return dict(label=label, theta=theta, se=se,
                ci_lo=ci_lo, ci_hi=ci_hi, pval=pval,
                r2_g=r2_g, r2_m=r2_m,
                Q_tilde=Q_tilde, P_tilde=P_tilde)

## 4. Estimate Across All Specifications

In [ ]:
results = []
for label, controls in specs.items():
    res = dml_no_crossfit(
        df_train, df_test,
        controls=controls,
        outcome=outcome,
        treatment=treatment,
        confidence_level=confidence_level,
        label=label,
    )
    results.append(res)
    print(f"{label:35s}  θ̂={res['theta']:+.4f}  "
          f"[{res['ci_lo']:+.4f}, {res['ci_hi']:+.4f}]  "
          f"R²_g={res['r2_g']:.3f}  R²_m={res['r2_m']:.3f}")

df_results = pd.DataFrame(
    [{k: v for k, v in r.items() if k not in ("Q_tilde", "P_tilde")} for r in results]
).set_index("label")

## 5. Results Table

In [ ]:
alpha = 1 - confidence_level
df_display = df_results[["theta", "se", "ci_lo", "ci_hi", "pval", "r2_g", "r2_m"]].copy()
df_display.columns = [
    "θ̂", "SE",
    f"[{alpha/2*100:.0f}%", f"{(1-alpha/2)*100:.0f}%]",
    "p-value", "R² outcome", "R² treatment"
]
df_display.index.name = "Controls"
df_display.round(4)

## 6. Forest Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
y = np.arange(len(df_results))

ax.errorbar(
    x=df_results["theta"],
    y=y,
    xerr=[df_results["theta"] - df_results["ci_lo"],
          df_results["ci_hi"] - df_results["theta"]],
    fmt="o", color=palette[0], ecolor=palette[1],
    capsize=5, capthick=2, linewidth=2, markersize=8,
    label=f"{confidence_level*100:.0f}% CI"
)
ax.axvline(0, color="gray", linestyle="--", linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(df_results.index, fontsize=10)
ax.set_xlabel("Estimated causal effect of price on quantity  (θ̂)", fontsize=11)
ax.set_title("DML Average Treatment Effect – No Cross-Fitting", fontsize=12)
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.4)
plt.tight_layout()
plt.show()

## 7. Second-Stage Diagnostics (Preferred Specification)

We inspect the partialled-out residuals and the second-stage regression
for the preferred specification (Tabular + Sim).

In [ ]:
preferred_idx = list(specs.keys()).index("Tabular + Sim")
preferred = results[preferred_idx]
Q_tilde = preferred["Q_tilde"]
P_tilde = preferred["P_tilde"]
theta_hat = preferred["theta"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residual distributions
axes[0].hist(Q_tilde, bins=60, color=palette[0], alpha=0.7, label="Q̃  (outcome residual)")
axes[0].hist(P_tilde, bins=60, color=palette[1], alpha=0.7, label="P̃  (treatment residual)")
axes[0].set_xlabel("Residual")
axes[0].set_title("Distribution of Partialled-Out Residuals")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Second-stage scatter
axes[1].scatter(P_tilde, Q_tilde, alpha=0.15, s=8, color=palette[0])
p_grid = np.linspace(P_tilde.min(), P_tilde.max(), 100)
axes[1].plot(p_grid, theta_hat * p_grid, color=palette[1], lw=2,
             label=f"θ̂ = {theta_hat:.4f}")
axes[1].set_xlabel("P̃  (price residual)")
axes[1].set_ylabel("Q̃  (quantity residual)")
axes[1].set_title("DML Second Stage (Tabular + Sim)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Interpretation:**  
- $\hat{\theta}$ measures the causal effect of a one-unit increase in buy-box price ($P_{bb,t}$) on sales rank ($Q_t$).
- A **negative** coefficient means higher prices lead to lower sales (worse rank).
- $R^2$ values for the nuisance models reflect how much confounding variation is absorbed before the second stage. Richer controls (embeddings, similarities) reduce omitted-variable bias.
- Confidence intervals use clustered standard errors (by ASIN) to account for serial correlation within products.